# Generic TFL Library — Phase 1 baseline
This notebook is offline after GitHub clone/install: no target SAP, API key or Drive required. It compares results to the teammate's published **in-library** benchmark, not independent clinical validation.

If upgrading from a previous notebook version, reopen this branch's notebook and run its setup cell first. The setup cell explicitly adds the checkout's src directory to the current Python kernel.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(ROOT)])
else:
    current = subprocess.check_output(['git', '-C', str(ROOT), 'branch', '--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{ROOT}[notebook,test]'])
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())

# Editable-install .pth files may not be processed by an already running Colab kernel.
# Import directly from the checkout, without requiring a runtime restart.
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent' / 'tools' / 'generic_library.py').is_file(), f'Generic agent source missing: {SRC}'
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
# Avoid silently mixing a previously imported main-branch package with this branch.
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Another tfl_agent checkout is already imported. Restart the Colab session and run all cells.')
import tfl_agent
from tfl_agent.tools.generic_library import load_generic_library
from tfl_agent.agent.generic_retriever import retrieve_generic_baseline
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC), tfl_agent.__file__
print('Agent package loaded:', tfl_agent.__file__)
print('Generic retrieval imports OK')


In [ ]:
from tfl_agent.tools.generic_library import load_generic_library
from tfl_agent.agent.generic_retriever import retrieve_generic_baseline
import json
library = load_generic_library(ROOT / 'generic_template_library')
print(json.dumps(library.summary(), indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
examples = [('Demographic and Baseline Characteristics – Safety Analysis Set', 'table'), ('Treatment-emergent adverse events by System Organ Class and Preferred Term', 'table'), ('Summary of Progression Free Survival – ITT Analysis Set', 'table')]
for title, kind in examples:
    r = retrieve_generic_baseline({'original_title': title, 'output_type': kind}, library)
    print(title, '\n', r['status'], r['template_id'], r['preset_id'], '\n', r['explanation'])
    display(pd.DataFrame(r['candidates']).head(3))

In [ ]:
from tfl_agent.agent.generic_evaluation import evaluate
summary, rows = evaluate(ROOT, library)
reference = json.loads((library.root / 'qa/matching_eval/Summary.json').read_text())
for key in ('entries_total','with_ground_truth','correct','accuracy','outcomes','by_output_type'):
    assert summary[key] == reference[key], f'Baseline mismatch: {key}'
print(json.dumps(summary, indent=2))
display(pd.DataFrame(rows).query("outcome not in ['correct', 'no_ground_truth']"))

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pytest', '-q', 'tests/test_generic_library_phase1.py'], cwd=ROOT)
subprocess.check_call([sys.executable, '-m', 'tfl_agent.agent.generic_evaluation', '--repo-root', str(ROOT), '--out', '/content/generic_phase1'], cwd=ROOT)
print('Audit output: /content/generic_phase1')